# DeepFashion Visual Features

**Owner:** Dasith  
**Purpose:** Build the visual feature learning component using CLIP image embeddings and transfer-learning-based fashion attribute classification.

This notebook consumes the manifests and label mapping produced by `notebooks/preprocessing/02_deepfashion_visual_preprocessing.ipynb`. Long-running CLIP extraction and classifier training are disabled by default. No cells in this notebook should be run automatically.

In [1]:
from pathlib import Path
import json
import random
import time
from contextlib import nullcontext

import os

# Disable Hugging Face Xet transfer because the download is repeatedly stalling
os.environ["HF_HUB_DISABLE_XET"] = "1"

print("HF Xet disabled:", os.environ["HF_HUB_DISABLE_XET"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

RANDOM_STATE = 42
FAST_DEV_RUN = False

RUN_CLIP_MODEL_TEST = False
RUN_CLIP_EXTRACTION = False
RUN_CLASSIFIER_TRAINING = True
FINAL_EVALUATION = False

BATCH_SIZE = 32
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
CLIP_TEST_IMAGES = 3
SIMILARITY_TOP_K = 5
NUM_WORKERS = 0
CHECKPOINT_INTERVAL = 768
runtime_timings = {'clip_extraction': 0.0, 'classifier_training': 0.0, 'final_evaluation': 0.0}
runtime_counters = {'clip_images': 0}
notebook_start_time = time.perf_counter()
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').is_file() and (candidate / 'data' / 'interim').is_dir():
            return candidate
    raise FileNotFoundError('Could not find a project root containing config.yaml and data/interim.')

PROJECT_ROOT = find_project_root()
INTERIM_ROOT = PROJECT_ROOT / 'data' / 'interim'
PROCESSED_FEATURE_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'visual_features'
MODEL_ROOT = PROJECT_ROOT / 'models' / 'visual'
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
FIGURES_ROOT = PROJECT_ROOT / 'outputs' / 'figures'
ARTIFACT_SUFFIX = '_dev' if FAST_DEV_RUN else ''
CLIP_EMBEDDINGS_PATH = PROCESSED_FEATURE_ROOT / f'deepfashion_clip_embeddings{ARTIFACT_SUFFIX}.npy'
CLIP_METADATA_PATH = PROCESSED_FEATURE_ROOT / f'deepfashion_clip_metadata{ARTIFACT_SUFFIX}.csv'
CLIP_CHECKPOINT_PATH = PROCESSED_FEATURE_ROOT / f'deepfashion_clip_embeddings{ARTIFACT_SUFFIX}.checkpoint.npz'
CLASSIFIER_CHECKPOINT_PATH = MODEL_ROOT / f'best_visual_attribute_model{ARTIFACT_SUFFIX}.pth'
CLASSIFIER_CONFIG_PATH = MODEL_ROOT / f'visual_model_config{ARTIFACT_SUFFIX}.json'
CLASSIFIER_METRICS_PATH = METRICS_ROOT / f'visual_model_results{ARTIFACT_SUFFIX}.csv'
TRAINING_CURVES_PATH = FIGURES_ROOT / f'dasith_visual_training_curves{ARTIFACT_SUFFIX}.png'
CONFUSION_MATRIX_PATH = FIGURES_ROOT / f'dasith_visual_confusion_matrix{ARTIFACT_SUFFIX}.png'
TEST_EXAMPLES_PATH = FIGURES_ROOT / f'dasith_visual_test_examples{ARTIFACT_SUFFIX}.png'
print(f'Project root: {PROJECT_ROOT}')
print(f'CLIP model test enabled: {RUN_CLIP_MODEL_TEST}')
print(f'CLIP extraction enabled: {RUN_CLIP_EXTRACTION}')
print(f'Classifier training enabled: {RUN_CLASSIFIER_TRAINING}')
print(f'Final test evaluation enabled: {FINAL_EVALUATION}')

HF Xet disabled: 1
Project root: D:\Deep Learning\Project\GITHUB\Explainable-Fashion-Design-AI
CLIP model test enabled: False
CLIP extraction enabled: False
Classifier training enabled: True
Final test evaluation enabled: False


## 1. Runtime and device configuration

The device is selected for later manual execution only. This cell reports the PyTorch and CUDA environment but does not start extraction or training.

In [2]:
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {DEVICE}')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
print(f'GPU name: {gpu_name}')

PyTorch version: 2.14.0+cpu
CUDA available: False
Device: cpu
GPU name: CPU only


## 2. Load and validate the preprocessing manifest

The manifest is metadata only: image files remain in their original location. Required columns and referenced paths are validated before any model workflow is enabled.

In [3]:
MANIFEST_PATH = INTERIM_ROOT / 'deepfashion_visual_manifest.csv'
MAPPING_PATH = INTERIM_ROOT / 'deepfashion_visual_label_mapping.json'
required_manifest_columns = {
    'image_id', 'image_path', 'target_label', 'target_id', 'target_name',
    'target_raw_id', 'target_raw_label',
}
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'Missing preprocessing manifest: {MANIFEST_PATH}')
if not MAPPING_PATH.is_file():
    raise FileNotFoundError(f'Missing label mapping: {MAPPING_PATH}')

def resolve_repo_image_path(path_value):
    path = Path(path_value)
    if path.is_absolute():
        raise ValueError(
            'Expected repository-relative image_path from preprocessing.'
        )
    resolved_path = (PROJECT_ROOT / path).resolve()
    if not resolved_path.is_file():
        raise FileNotFoundError(
            f'Referenced image is missing: {resolved_path}'
        )
    return resolved_path

manifest = pd.read_csv(MANIFEST_PATH)
missing_columns = required_manifest_columns.difference(manifest.columns)
if missing_columns:
    raise ValueError(
        'Preprocessing manifest does not satisfy the required semantic schema. '
        f'Missing columns: {sorted(missing_columns)}'
    )

manifest['resolved_image_path'] = manifest['image_path'].map(resolve_repo_image_path)
print(f'Manifest rows: {len(manifest):,}')
print(f'Validated image paths: {len(manifest):,}')

def deterministic_runtime_subset(frame, limit):
    if len(frame) <= limit:
        return frame.reset_index(drop=True)
    groups = []
    for _, group in frame.groupby('target_id', sort=True):
        groups.append(group.head(max(1, limit // frame['target_id'].nunique())))
    subset = pd.concat(groups, ignore_index=True).head(limit)
    return subset.reset_index(drop=True)


runtime_manifest = deterministic_runtime_subset(manifest, 96) if FAST_DEV_RUN else manifest
if FAST_DEV_RUN:
    print(f'FAST_DEV_RUN enabled: using {len(runtime_manifest):,} deterministic CLIP images.')

label_mapping = json.loads(MAPPING_PATH.read_text(encoding='utf-8'))
TARGET_NAME = label_mapping.get('target_name', 'unknown')
TARGET_COLUMN = label_mapping.get('target_column', 'unknown')
ID_TO_LABEL = {int(key): value for key, value in label_mapping.get('id_to_label', {}).items()}
LABEL_TO_ID = label_mapping.get('label_to_id', {})
if TARGET_NAME != 'lower_fabric':
    raise ValueError(f'Expected lower_fabric target, found: {TARGET_NAME}')
if manifest['target_name'].nunique() != 1 or manifest['target_name'].iloc[0] != TARGET_NAME:
    raise ValueError('Manifest target_name does not match the read-only preprocessing mapping.')
if sorted(manifest['target_id'].unique().tolist()) != list(range(manifest['target_id'].nunique())):
    raise ValueError('Manifest target IDs must be contiguous and zero-based.')
print(f'Preprocessing target: {TARGET_NAME} ({TARGET_COLUMN})')
print(f"Number of classes: {manifest['target_id'].nunique()}")
display(manifest[['image_id', 'image_path', 'target_name', 'target_label', 'target_id']].head())

Manifest rows: 42,755
Validated image paths: 42,755
Preprocessing target: lower_fabric (texture/fabric_ann.txt__value_2)
Number of classes: 6


,image_id,image_path,target_name,target_label,target_id
0,MEN-Denim-id_00000080-01_7_additional.jpg,data/raw/deepfashion/images/MEN-Denim-id_00000...,lower_fabric,cotton,1
1,MEN-Denim-id_00000089-01_7_additional.jpg,data/raw/deepfashion/images/MEN-Denim-id_00000...,lower_fabric,cotton,1
2,MEN-Denim-id_00000089-02_7_additional.jpg,data/raw/deepfashion/images/MEN-Denim-id_00000...,lower_fabric,cotton,1
3,MEN-Denim-id_00000089-03_7_additional.jpg,data/raw/deepfashion/images/MEN-Denim-id_00000...,lower_fabric,cotton,1
4,MEN-Denim-id_00000089-04_7_additional.jpg,data/raw/deepfashion/images/MEN-Denim-id_00000...,lower_fabric,cotton,1


## 3. CLIP image embeddings

A pretrained Hugging Face CLIP image encoder provides reusable semantic visual features. Images are loaded lazily, processed in batches, and inference uses evaluation mode plus `torch.no_grad()`. The full extraction block runs only when `RUN_CLIP_EXTRACTION` is set to `True`.

In [4]:
import os

os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'

print('Hugging Face Hub timeouts configured; cached and partial files will be preserved.')

Hugging Face Hub timeouts configured; cached and partial files will be preserved.


In [5]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"


def load_clip_model(model_name, device):
    try:
        from transformers import CLIPModel, CLIPProcessor
        processor = CLIPProcessor.from_pretrained(model_name)
        model = CLIPModel.from_pretrained(model_name)
        model = model.to(device)
        print('CLIP model loaded successfully.')
        return processor, model
    except Exception as error:
        print(f'CLIP model loading failed: {type(error).__name__}: {error}')
        print('The download may be incomplete due to a network timeout or another Hub error.')
        print('Do not delete the Hugging Face cache. Rerun this cell to resume/reuse cached files.')
        return None, None


def clip_image_embedding(output):
    if isinstance(output, torch.Tensor):
        embeddings = output
    elif hasattr(output, 'pooler_output') and output.pooler_output is not None:
        embeddings = output.pooler_output
    elif hasattr(output, 'image_embeds') and output.image_embeds is not None:
        embeddings = output.image_embeds
    else:
        raise TypeError(f'Unsupported CLIP image feature output type: {type(output).__name__}')
    return F.normalize(embeddings, p=2, dim=-1)


if RUN_CLIP_MODEL_TEST or RUN_CLIP_EXTRACTION:
    clip_setup_start = time.perf_counter()
    clip_processor, clip_model = load_clip_model(CLIP_MODEL_NAME, DEVICE)
    print(f'[1/4] CLIP model setup: {time.perf_counter() - clip_setup_start:.2f}s')
else:
    clip_processor, clip_model = None, None
    print('CLIP loading skipped. Enable RUN_CLIP_MODEL_TEST or RUN_CLIP_EXTRACTION to load it.')


class ClipImageDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row['resolved_image_path']) as image:
            return {'image': image.convert('RGB'), 'image_id': row['image_id'], 'image_path': row['image_path']}


def clip_collate(batch):
    images = [item['image'] for item in batch]
    encoded = clip_processor(images=images, return_tensors='pt')
    return encoded, [item['image_id'] for item in batch], [item['image_path'] for item in batch]


if RUN_CLIP_MODEL_TEST:
    if clip_processor is None or clip_model is None:
        print('CLIP model test skipped because the CLIP model is unavailable.')
    else:
        clip_test_frame = runtime_manifest.head(min(CLIP_TEST_IMAGES, len(runtime_manifest)))
        clip_test_loader = DataLoader(ClipImageDataset(clip_test_frame), batch_size=CLIP_TEST_IMAGES, shuffle=False, collate_fn=clip_collate, num_workers=NUM_WORKERS)
        with torch.no_grad():
            test_inputs, test_ids, test_paths = next(iter(clip_test_loader))
            test_inputs = {key: value.to(DEVICE) for key, value in test_inputs.items()}
            image_output = clip_model.get_image_features(**test_inputs)
            test_embeddings = clip_image_embedding(image_output)
        for filename, embedding in zip(test_ids, test_embeddings):
            print(f'filename={filename}; embedding_shape={tuple(embedding.shape)}; dtype={embedding.dtype}')


class ClipEmbeddingDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row['resolved_image_path']) as image:
            return image.convert('RGB'), row['image_id'], row['image_path']


def embedding_collate(batch):
    images, image_ids, image_paths = zip(*batch)
    encoded = clip_processor(images=list(images), return_tensors='pt')
    return encoded, list(image_ids), list(image_paths)


def extract_clip_embeddings(frame, batch_size=BATCH_SIZE):
    if clip_processor is None or clip_model is None:
        print('CLIP model is not available. Load the model successfully before continuing.')
        return None, None
    PROCESSED_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
    checkpoint_path = CLIP_CHECKPOINT_PATH
    completed = {}
    if checkpoint_path.is_file():
        try:
            checkpoint = np.load(checkpoint_path, allow_pickle=False)
            checkpoint_ids = checkpoint['image_id'].astype(str).tolist()
            checkpoint_paths = checkpoint['image_path'].astype(str).tolist()
            checkpoint_embeddings = checkpoint['embeddings']
            if checkpoint_embeddings.ndim == 2 and len(checkpoint_ids) == len(checkpoint_embeddings):
                completed = {
                    image_id: (image_path, embedding)
                    for image_id, image_path, embedding in zip(checkpoint_ids, checkpoint_paths, checkpoint_embeddings)
                }
                print(f'Resuming CLIP extraction from {len(completed):,} checkpointed images.')
        except (OSError, ValueError, KeyError) as error:
            print(f'Ignoring invalid CLIP checkpoint: {error}')
    pending_frame = frame[~frame['image_id'].astype(str).isin(completed)].reset_index(drop=True)
    loader = DataLoader(ClipEmbeddingDataset(pending_frame), batch_size=batch_size, shuffle=False, collate_fn=embedding_collate, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    extraction_start = time.perf_counter()
    processed_since_checkpoint = 0
    clip_model.eval()
    with torch.inference_mode():
        progress = tqdm(loader, desc='Extracting CLIP embeddings', unit='batch')
        for inputs, image_ids, image_paths in progress:
            inputs = {key: value.to(DEVICE, non_blocking=True) for key, value in inputs.items()}
            image_output = clip_model.get_image_features(**inputs)
            embeddings = clip_image_embedding(image_output).float().cpu()
            for image_id, image_path, embedding in zip(image_ids, image_paths, embeddings.numpy()):
                completed[str(image_id)] = (image_path, embedding)
            processed_since_checkpoint += len(image_ids)
            elapsed = time.perf_counter() - extraction_start
            processed = len(completed)
            rate = processed / max(elapsed, 1e-6)
            progress.set_postfix(images=f'{processed:,}/{len(frame):,}', elapsed=f'{elapsed:.1f}s', rate=f'{rate:.1f}/s', eta=f'{(len(frame) - processed) / max(rate, 1e-6):.1f}s')
            if processed_since_checkpoint >= CHECKPOINT_INTERVAL:
                ordered = frame[frame['image_id'].astype(str).isin(completed)]
                np.savez(checkpoint_path, embeddings=np.stack([completed[str(image_id)][1] for image_id in ordered['image_id']]), image_id=ordered['image_id'].astype(str).to_numpy(), image_path=ordered['image_path'].astype(str).to_numpy())
                processed_since_checkpoint = 0
    ordered = frame
    matrix = np.stack([completed[str(image_id)][1] for image_id in ordered['image_id']]) if len(ordered) else np.empty((0, 0), dtype=np.float32)
    metadata = ordered[['image_id', 'image_path']].reset_index(drop=True).copy()
    runtime_counters['clip_images'] = len(ordered)
    runtime_timings['clip_extraction'] = time.perf_counter() - extraction_start
    np.savez(checkpoint_path, embeddings=matrix, image_id=metadata['image_id'].astype(str).to_numpy(), image_path=metadata['image_path'].astype(str).to_numpy())
    return matrix, metadata


def validate_embedding_matrix(matrix):
    if matrix.ndim != 2 or matrix.shape[0] != len(runtime_manifest):
        raise ValueError(f'Unexpected embedding shape: {matrix.shape}')
    if not np.isfinite(matrix).all():
        raise ValueError('Embeddings contain NaN or Inf values.')
    print(f'Embedding matrix shape: {matrix.shape}; dtype: {matrix.dtype}; dimension: {matrix.shape[1]}')


if RUN_CLIP_EXTRACTION:
    if clip_processor is None or clip_model is None:
        print('CLIP model is not available. Load the model successfully before continuing.')
    else:
        clip_embeddings, clip_metadata = extract_clip_embeddings(runtime_manifest)
        if clip_embeddings is not None:
            validate_embedding_matrix(clip_embeddings)
            PROCESSED_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
            np.save(CLIP_EMBEDDINGS_PATH, clip_embeddings)
            clip_metadata.to_csv(CLIP_METADATA_PATH, index=False)
            CLIP_CHECKPOINT_PATH.unlink(missing_ok=True)
            print(f'[2/4] CLIP embedding extraction: {runtime_timings["clip_extraction"]:.2f}s')
            print(f'Saved CLIP outputs to {PROCESSED_FEATURE_ROOT}')
else:
    print('Full CLIP extraction is disabled. Set RUN_CLIP_EXTRACTION=True to run it manually.')

CLIP loading skipped. Enable RUN_CLIP_MODEL_TEST or RUN_CLIP_EXTRACTION to load it.
Full CLIP extraction is disabled. Set RUN_CLIP_EXTRACTION=True to run it manually.


## 4. Cosine similarity retrieval

When saved embeddings are available, this section retrieves visually similar real images using cosine similarity. It does not require loading images into RAM; only the embedding matrix and selected image previews are used.

In [6]:
def retrieve_similar_images(embeddings, metadata, query_index=0, top_k=SIMILARITY_TOP_K):
    normalized = F.normalize(torch.as_tensor(embeddings), p=2, dim=1)
    scores = normalized @ normalized[query_index]
    scores[query_index] = -torch.inf
    values, indices = torch.topk(scores, k=min(top_k, len(metadata) - 1))
    results = metadata.iloc[indices.numpy()].copy()
    results['cosine_similarity'] = values.numpy()
    return results


if clip_processor is None or clip_model is None:
    print('CLIP model is not available. Load the model successfully before continuing.')
elif 'clip_embeddings' in globals() and len(clip_embeddings) > 1:
    similarity_results = retrieve_similar_images(clip_embeddings, clip_metadata)
    display(similarity_results)
else:
    print('Similarity retrieval is skipped until CLIP embeddings have been generated.')

CLIP model is not available. Load the model successfully before continuing.


## 5. Classification data and lazy loaders

The classifier uses the exact train, validation, and test manifests from preprocessing. Images are decoded lazily in a PyTorch `Dataset`; the untouched test set is not used during training or model selection.

In [7]:
from torchvision import transforms
from torchvision.models import ResNet50_Weights
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix

torch.manual_seed(RANDOM_STATE)
loader_generator = torch.Generator()
loader_generator.manual_seed(RANDOM_STATE)

TRAIN_PATH = INTERIM_ROOT / 'deepfashion_visual_train.csv'
VALIDATION_PATH = INTERIM_ROOT / 'deepfashion_visual_validation.csv'
TEST_PATH = INTERIM_ROOT / 'deepfashion_visual_test.csv'
for split_path in (TRAIN_PATH, VALIDATION_PATH, TEST_PATH):
    if not split_path.is_file():
        raise FileNotFoundError(f'Missing split file: {split_path}')

train_frame = pd.read_csv(TRAIN_PATH)
validation_frame = pd.read_csv(VALIDATION_PATH)
test_frame = pd.read_csv(TEST_PATH)
if FAST_DEV_RUN:
    train_frame = deterministic_runtime_subset(train_frame, 128)
    validation_frame = deterministic_runtime_subset(validation_frame, 64)
    test_frame = deterministic_runtime_subset(test_frame, 64)
    print(f'FAST_DEV_RUN enabled: train={len(train_frame):,}, validation={len(validation_frame):,}, test={len(test_frame):,}.')
for split_name, frame in {'train': train_frame, 'validation': validation_frame, 'test': test_frame}.items():
    if not required_manifest_columns.issubset(frame.columns):
        raise ValueError(f'{split_name} split is missing semantic manifest columns.')
    frame['resolved_image_path'] = frame['image_path'].map(resolve_repo_image_path)
    if set(frame['target_id'].unique()) != set(manifest['target_id'].unique()):
        raise ValueError(f'{split_name} split does not contain the same modeled class IDs as the manifest.')

n_classes = int(train_frame['target_id'].nunique())
train_counts = train_frame['target_id'].value_counts().sort_index()
majority_id = int(train_counts.idxmax())

resnet_weights = ResNet50_Weights.DEFAULT
resnet_eval_transform = resnet_weights.transforms()
resnet_train_transform = transforms.Compose([
    transforms.Lambda(lambda image: image.convert('RGB')),
    transforms.RandomResizedCrop(resnet_weights.transforms().crop_size[0], scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(resnet_weights.transforms().mean, resnet_weights.transforms().std),
])
print(f'Classes: {n_classes}; majority class: {ID_TO_LABEL.get(majority_id, majority_id)}')
print('ResNet50 uses official ImageNet-compatible pretrained-weight transforms; CLIP normalization is not used.')

class FashionAttributeDataset(torch.utils.data.Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row['resolved_image_path']) as image:
            tensor = self.transform(image.convert('RGB'))
        return tensor, int(row['target_id']), row['image_id'], row['image_path']

train_loader = DataLoader(FashionAttributeDataset(train_frame, resnet_train_transform), batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
validation_loader = DataLoader(FashionAttributeDataset(validation_frame, resnet_eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(FashionAttributeDataset(test_frame, resnet_eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

Classes: 6; majority class: denim
ResNet50 uses official ImageNet-compatible pretrained-weight transforms; CLIP normalization is not used.


## 6. ResNet50 transfer-learning classifier

The default classifier is pretrained ResNet50 with only its final layer replaced for the observed class count. Pretrained layers are frozen initially. Cross-entropy uses class weights only when the observed training distribution is sufficiently imbalanced; AdamW is used for optimization.

In [8]:
from torchvision.models import resnet50


def make_resnet50_classifier(num_classes):
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)


train_max_count = int(train_counts.max())
train_min_count = int(train_counts.min())
train_imbalance_ratio = train_max_count / train_min_count if train_min_count else np.inf
minority_to_majority_ratio = train_min_count / train_max_count if train_max_count else 0.0
USE_CLASS_WEIGHTS = bool(train_imbalance_ratio >= 2.0 or minority_to_majority_ratio < 0.5)
if USE_CLASS_WEIGHTS:
    class_weight_values = train_counts.reindex(range(n_classes))
    if class_weight_values.isna().any() or (class_weight_values <= 0).any():
        raise ValueError('Every modeled class must be present in the training split before calculating weights.')
    class_weights = {
        int(class_id): float(len(train_frame) / (n_classes * count))
        for class_id, count in class_weight_values.items()
    }
    loss_weights = torch.tensor([class_weights[class_id] for class_id in range(n_classes)], dtype=torch.float32, device=DEVICE)
else:
    class_weights = {}
    loss_weights = None
print(f'Train max/min frequency ratio: {train_imbalance_ratio:.3f}')
print(f'Minority-to-majority ratio: {minority_to_majority_ratio:.3f}')
print(f'Use class weights: {USE_CLASS_WEIGHTS}; source: training split only')


def build_classifier_training_objects():
    model = make_resnet50_classifier(n_classes)
    criterion = torch.nn.CrossEntropyLoss(weight=loss_weights)
    optimizer = torch.optim.AdamW((parameter for parameter in model.parameters() if parameter.requires_grad), lr=LEARNING_RATE)
    return model, criterion, optimizer

Train max/min frequency ratio: 187.134
Minority-to-majority ratio: 0.005
Use class weights: True; source: training split only


## 7. Optional classifier training

Training uses only the train split, while validation metrics select the best checkpoint. Test metrics, confusion matrices, and test-example predictions are gated by `FINAL_EVALUATION=True` and are produced only from the finalized, frozen best validation checkpoint. No test metrics or majority baseline are calculated during normal development.

In [9]:
def classification_metrics(true_labels, predicted_labels):
    return {
        'accuracy': accuracy_score(true_labels, predicted_labels),
        'macro_precision': precision_score(true_labels, predicted_labels, average='macro', zero_division=0),
        'macro_recall': recall_score(true_labels, predicted_labels, average='macro', zero_division=0),
        'macro_f1': f1_score(true_labels, predicted_labels, average='macro', zero_division=0),
        'weighted_f1': f1_score(true_labels, predicted_labels, average='weighted', zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(true_labels, predicted_labels),
    }


def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, true_labels, predicted_labels = 0.0, [], []
    context = nullcontext() if training else torch.no_grad()
    with context:
        for images, labels, _, _ in tqdm(loader, desc='Training' if training else 'Evaluation'):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            true_labels.extend(labels.detach().cpu().numpy())
            predicted_labels.extend(logits.argmax(dim=1).detach().cpu().numpy())
    average_loss = total_loss / len(loader.dataset)
    metrics = classification_metrics(true_labels, predicted_labels)
    metrics.update({'loss': average_loss, 'true': np.asarray(true_labels), 'predicted': np.asarray(predicted_labels)})
    return metrics


def train_and_evaluate_classifier():
    training_start = time.perf_counter()
    model, criterion, optimizer = build_classifier_training_objects()
    history, best_val_f1, best_epoch = [], -np.inf, None
    MODEL_ROOT.mkdir(parents=True, exist_ok=True)
    checkpoint_path = CLASSIFIER_CHECKPOINT_PATH
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start = time.perf_counter()
        train_metrics = run_epoch(model, train_loader, criterion, optimizer)
        validation_metrics = run_epoch(model, validation_loader, criterion)
        epoch_elapsed = time.perf_counter() - epoch_start
        record = {'epoch': epoch, 'training_loss': train_metrics['loss'], 'validation_loss': validation_metrics['loss'], 'validation_accuracy': validation_metrics['accuracy'], 'validation_macro_f1': validation_metrics['macro_f1'], 'epoch_elapsed_seconds': epoch_elapsed}
        history.append(record)
        print(f"Epoch {epoch}/{NUM_EPOCHS} | batches={len(train_loader):,} | train_loss={train_metrics['loss']:.4f} | validation_loss={validation_metrics['loss']:.4f} | validation_accuracy={validation_metrics['accuracy']:.4f} | validation_macro_f1={validation_metrics['macro_f1']:.4f} | epoch_elapsed={epoch_elapsed:.1f}s | total_elapsed={time.perf_counter() - training_start:.1f}s")
        if validation_metrics['macro_f1'] > best_val_f1:
            best_val_f1 = validation_metrics['macro_f1']
            best_epoch = epoch
            torch.save({'model_state_dict': model.state_dict(), 'target_name': TARGET_NAME, 'target_column': TARGET_COLUMN, 'num_classes': n_classes, 'class_labels': ID_TO_LABEL, 'best_validation_macro_f1': best_val_f1, 'best_epoch': best_epoch}, checkpoint_path)

    best_model = make_resnet50_classifier(n_classes)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    if checkpoint['target_name'] != TARGET_NAME or checkpoint['num_classes'] != n_classes:
        raise ValueError('Best checkpoint metadata does not match the current semantic target or class count.')
    best_model.load_state_dict(checkpoint['model_state_dict'])
    best_model.eval()
    for parameter in best_model.parameters():
        parameter.requires_grad = False

    test_metrics = None
    test_metric_values = None
    if FINAL_EVALUATION:
        print('Evaluating the finalized, frozen best validation checkpoint on the untouched test split.')
        evaluation_start = time.perf_counter()
        test_metrics = run_epoch(best_model, test_loader, criterion)
        runtime_timings['final_evaluation'] = time.perf_counter() - evaluation_start
        test_metric_values = classification_metrics(test_metrics['true'], test_metrics['predicted'])
        test_metric_values.update({'split': 'test', 'model': 'resnet50_best_validation_checkpoint'})
    runtime_timings['classifier_training'] = time.perf_counter() - training_start
    return best_model, pd.DataFrame(history), test_metrics, test_metric_values, checkpoint


if RUN_CLASSIFIER_TRAINING:
    classifier_model, training_history, test_metrics, test_metric_values, best_checkpoint = train_and_evaluate_classifier()
    print(f'[3/4] Visual classifier training: {runtime_timings["classifier_training"]:.2f}s; average epoch: {training_history["epoch_elapsed_seconds"].mean():.2f}s')
    MODEL_ROOT.mkdir(parents=True, exist_ok=True)
    CLASSIFIER_CONFIG_PATH.write_text(json.dumps({'random_state': RANDOM_STATE, 'batch_size': BATCH_SIZE, 'num_epochs': NUM_EPOCHS, 'learning_rate': LEARNING_RATE, 'target_name': TARGET_NAME, 'target_column': TARGET_COLUMN, 'num_classes': n_classes, 'use_class_weights': USE_CLASS_WEIGHTS, 'class_weights': class_weights, 'class_weights_source': 'training split only', 'resnet_preprocessing': 'ResNet50_Weights.DEFAULT.transforms()', 'device': str(DEVICE), 'final_evaluation': FINAL_EVALUATION}, indent=2), encoding='utf-8')
    if FINAL_EVALUATION:
        print(f'[4/4] Final evaluation: {runtime_timings["final_evaluation"]:.2f}s')
        METRICS_ROOT.mkdir(parents=True, exist_ok=True)
        pd.DataFrame([test_metric_values]).to_csv(CLASSIFIER_METRICS_PATH, index=False)
    else:
        print('Final test evaluation is disabled; no test metrics were calculated or saved.')
else:
    print('Classifier training is disabled. No checkpoint, test metrics, or training curves are produced.')

Training:   0%|          | 0/1069 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/134 [00:00<?, ?it/s]

Epoch 1/5 | batches=1,069 | train_loss=1.6070 | validation_loss=1.5224 | validation_accuracy=0.4989 | validation_macro_f1=0.2657 | epoch_elapsed=2467.6s | total_elapsed=2468.0s


Training:   0%|          | 0/1069 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/134 [00:00<?, ?it/s]

...

Evaluation:   0%|          | 0/134 [00:00<?, ?it/s]

Epoch 4/5 | batches=1,069 | train_loss=1.3385 | validation_loss=1.4126 | validation_accuracy=0.4784 | validation_macro_f1=0.2696 | epoch_elapsed=2752.3s | total_elapsed=11820.8s


Training:   0%|          | 0/1069 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/134 [00:00<?, ?it/s]

Epoch 5/5 | batches=1,069 | train_loss=1.2940 | validation_loss=1.3970 | validation_accuracy=0.4854 | validation_macro_f1=0.2956 | epoch_elapsed=2427.6s | total_elapsed=14248.5s
[3/4] Visual classifier training: 14249.20s; average epoch: 2849.52s
Final test evaluation is disabled; no test metrics were calculated or saved.


## 8. Optional training plots, confusion matrix, and test examples

These visualizations are created only from real trained outputs. Matplotlib is used exclusively; seaborn is not used.

In [10]:
if RUN_CLASSIFIER_TRAINING and FINAL_EVALUATION:
    FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(training_history['epoch'], training_history['training_loss'], label='training loss')
    axes[0].plot(training_history['epoch'], training_history['validation_loss'], label='validation loss')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
    axes[1].plot(training_history['epoch'], training_history['validation_accuracy'], label='validation accuracy')
    axes[1].plot(training_history['epoch'], training_history['validation_macro_f1'], label='validation macro F1')
    axes[1].set_title('Validation metrics'); axes[1].set_xlabel('Epoch'); axes[1].legend()
    fig.tight_layout(); fig.savefig(TRAINING_CURVES_PATH, dpi=150, bbox_inches='tight'); plt.show(); plt.close(fig)

    cm = confusion_matrix(test_metrics['true'], test_metrics['predicted'], labels=list(range(n_classes)))
    fig, ax = plt.subplots(figsize=(8, 7)); image = ax.imshow(cm, cmap='Blues'); fig.colorbar(image, ax=ax)
    ax.set_title('Test confusion matrix'); ax.set_xlabel('Predicted label ID'); ax.set_ylabel('True label ID')
    fig.tight_layout(); fig.savefig(CONFUSION_MATRIX_PATH, dpi=150, bbox_inches='tight'); plt.show(); plt.close(fig)

    sample_indices = np.random.default_rng(RANDOM_STATE).choice(len(test_frame), size=min(6, len(test_frame)), replace=False)
    sample_frame = test_frame.iloc[sample_indices]
    classifier_model.eval()
    fig, axes = plt.subplots(2, 3, figsize=(12, 8)); axes = np.asarray(axes).ravel()
    for ax in axes: ax.axis('off')
    with torch.no_grad():
        for ax, (_, row) in zip(axes, sample_frame.iterrows()):
            with Image.open(row['resolved_image_path']) as image:
                original = image.convert('RGB')
            logits = classifier_model(resnet_eval_transform(original).unsqueeze(0).to(DEVICE))
            probabilities = torch.softmax(logits, dim=1)[0]
            prediction = int(probabilities.argmax())
            confidence = float(probabilities[prediction])
            true_label = ID_TO_LABEL.get(int(row['target_id']), str(row['target_label']))
            predicted_label = ID_TO_LABEL.get(prediction, str(prediction))
            title = 'true_label={} | predicted_label={}\\nconfidence={:.3f}'.format(true_label, predicted_label, confidence)
            ax.imshow(original); ax.set_title(title, fontsize=9); ax.axis('off')
    fig.tight_layout(); fig.savefig(TEST_EXAMPLES_PATH, dpi=150, bbox_inches='tight'); plt.show(); plt.close(fig)
elif RUN_CLASSIFIER_TRAINING:
    print('Final evaluation is disabled; test visualizations are skipped.')
else:
    print('Training plots, confusion matrix, and test predictions are skipped until finalized training and evaluation are enabled.')

Final evaluation is disabled; test visualizations are skipped.


## 9. Output summary, limitations, and conclusion

When run manually with the gates enabled, this notebook writes only derived feature/model/metric artifacts. It never modifies raw DeepFashion files.

In [11]:
average_epoch_time = float(training_history['epoch_elapsed_seconds'].mean()) if 'training_history' in globals() else 0.0
clip_rate = runtime_counters['clip_images'] / runtime_timings['clip_extraction'] if runtime_timings['clip_extraction'] > 0 else 0.0
print(f'Runtime summary | CLIP extraction: {runtime_timings["clip_extraction"]:.2f}s ({clip_rate:.2f} images/s) | classifier training: {runtime_timings["classifier_training"]:.2f}s | average epoch: {average_epoch_time:.2f}s | final evaluation: {runtime_timings["final_evaluation"]:.2f}s | total: {time.perf_counter() - notebook_start_time:.2f}s')

display(Markdown('''### Output summary

- Semantic target: **lower_fabric**, using the preprocessing notebook's human-readable labels and explicit NA/rare-class policy.
- CLIP outputs are saved only when `RUN_CLIP_EXTRACTION=True`; CLIP embeddings are L2-normalized and support Tensor or ModelOutput return types.
- Classifier outputs are saved only when `RUN_CLASSIFIER_TRAINING=True`; ResNet50 uses official ImageNet-compatible preprocessing, separate from CLIP preprocessing.
- Training uses the train split and model selection uses the validation split. The test split remains untouched unless `FINAL_EVALUATION=True` for a finalized, frozen checkpoint.

### Limitations

- No model performance can be claimed while the execution gates remain disabled.
- ID `7` is a real `NA` annotation and rare classes may have been excluded before supervised splitting; these choices affect generalization.
- Frozen-backbone ResNet50 transfer learning may underfit fashion-specific attributes; fine-tuning should be guided by validation results.
- CLIP and ResNet50 require pretrained weights and an existing configured environment; this notebook does not install packages or require authentication.

### Conclusion

This notebook provides a gated, memory-conscious CLIP feature pipeline and a ResNet50 transfer-learning classifier for the semantic `lower_fabric` task. It treats preprocessing manifests and mappings as read-only contracts, uses validation for development decisions, preserves the untouched test split, and evaluates test performance only for an explicitly enabled finalized evaluation.'''))

Runtime summary | CLIP extraction: 0.00s (0.00 images/s) | classifier training: 14249.20s | average epoch: 2849.52s | final evaluation: 0.00s | total: 14286.27s


### Output summary

- Semantic target: **lower_fabric**, using the preprocessing notebook's human-readable labels and explicit NA/rare-class policy.
- CLIP outputs are saved only when `RUN_CLIP_EXTRACTION=True`; CLIP embeddings are L2-normalized and support Tensor or ModelOutput return types.
- Classifier outputs are saved only when `RUN_CLASSIFIER_TRAINING=True`; ResNet50 uses official ImageNet-compatible preprocessing, separate from CLIP preprocessing.
- Training uses the train split and model selection uses the validation split. The test split remains untouched unless `FINAL_EVALUATION=True` for a finalized, frozen checkpoint.

### Limitations

- No model performance can be claimed while the execution gates remain disabled.
- ID `7` is a real `NA` annotation and rare classes may have been excluded before supervised splitting; these choices affect generalization.
- Frozen-backbone ResNet50 transfer learning may underfit fashion-specific attributes; fine-tuning should be guided by validation results.
- CLIP and ResNet50 require pretrained weights and an existing configured environment; this notebook does not install packages or require authentication.

### Conclusion

This notebook provides a gated, memory-conscious CLIP feature pipeline and a ResNet50 transfer-learning classifier for the semantic `lower_fabric` task. It treats preprocessing manifests and mappings as read-only contracts, uses validation for development decisions, preserves the untouched test split, and evaluates test performance only for an explicitly enabled finalized evaluation.